# 14 — MuZero + Gymnasium

**Before:** MuZero notebooks **11–13**.

**This notebook:** intro to classic control (CartPole). Requires `pip install -e ".[dev,atari]"`.

**Learning objectives**

- Wrap a Gymnasium environment for MuZero.
- Handle vector observations (e.g. CartPole).
- See how the same algorithm generalizes beyond board games.
- Run the CartPole demo cell successfully.

**Online course:** run cells top-to-bottom. In setup, keep `RUN_TRAIN=False` until you want a long training run. Set `PLAY_INTERACTIVE=True` only to play in the terminal.

**Install:** `pip install -e ".[dev,atari]"` from the AlphaChild repo root.

Curriculum: `docs/ONLINE_COURSE.md`


In [ ]:
# --- Course setup (AlphaChild repo root) ---
import sys
from pathlib import Path


def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "alphazero").is_dir() and (base / "1.TicTacToe.ipynb").is_file():
            return base
        if (base / "alphazero").is_dir() and (base / "pyproject.toml").is_file():
            return base
    return Path.cwd()


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from alphazero.notebook_utils import checkpoint_path

RUN_TRAIN = False           # True: run self-play training (slow — minutes+)
PLAY_INTERACTIVE = False    # True: human vs AI in terminal (needs keyboard input)
DEMO_SEARCHES = 100         # MCTS searches for demos; increase when curious

print("ROOT", ROOT.resolve())
print("RUN_TRAIN", RUN_TRAIN, "| PLAY_INTERACTIVE", PLAY_INTERACTIVE, "| DEMO_SEARCHES", DEMO_SEARCHES)


In [ ]:
import numpy as np
print(np.__version__)

try:
    import gymnasium as gym
    print(gym.__version__)
except ImportError:
    print("Install gymnasium: pip install gymnasium")
    raise

from collections import deque


In [ ]:
class GymGame:
    """Gymnasium env with the same API as TicTacToe in the earlier notebooks."""

    def __init__(self, env_id="CartPole-v1", frame_skip=1, max_episode_steps=None):
        self.env_id = env_id
        self.frame_skip = frame_skip
        self._last_reward = 0.0
        self._last_done = False

        self.env = gym.make(env_id)
        if max_episode_steps:
            self.env = gym.wrappers.TimeLimit(self.env, max_episode_steps=max_episode_steps)

        self.action_size = int(self.env.action_space.n)
        obs_shape = self.env.observation_space.shape
        self.obs_kind = "vector" if len(obs_shape) == 1 else "image"
        self.row_count = 1
        self.column_count = max(4, int(np.ceil(obs_shape[0] / 4))) if self.obs_kind == "vector" else 6

    def __repr__(self):
        return f"GymGame({self.env_id!r})"

    def get_initial_state(self):
        obs, _ = self.env.reset()
        self._last_reward = 0.0
        self._last_done = False
        return np.asarray(obs, dtype=np.float32)

    def get_next_state(self, state, action, player=1):
        obs = state
        total_reward = 0.0
        terminated = truncated = False
        for _ in range(self.frame_skip):
            obs, reward, terminated, truncated, _ = self.env.step(int(action))
            total_reward += float(reward)
            if terminated or truncated:
                break
        self._last_reward = total_reward
        self._last_done = terminated or truncated
        return np.asarray(obs, dtype=np.float32)

    def get_valid_moves(self, state):
        return np.ones(self.action_size, dtype=np.uint8)

    def get_value_and_terminated(self, state, action):
        return self._last_reward, self._last_done

    def get_opponent(self, player):
        return player

    def get_opponent_value(self, value):
        return value

    def change_perspective(self, state, player):
        return state

    def get_encoded_state(self, state):
        encoded = np.asarray(state, dtype=np.float32)
        if encoded.ndim == 1:
            encoded = encoded.reshape(1, -1)
        return encoded


In [ ]:
cartpole = GymGame("CartPole-v1", max_episode_steps=200)
state = cartpole.get_initial_state()
print(cartpole)
print("observation shape:", state.shape)
print("encoded shape:", cartpole.get_encoded_state(state).shape)
print("action_size:", cartpole.action_size)

for step in range(5):
    action = cartpole.env.action_space.sample()
    state = cartpole.get_next_state(state, action)
    reward, done = cartpole.get_value_and_terminated(state, action)
    print(f"step {step}: action={action}, reward={reward}, done={done}")
    if done:
        break


In [ ]:
class AtariGym(GymGame):
    """Atari with grayscale frames and stacking — same interface, visual observations."""

    def __init__(self, env_id="ALE/Pong-v5", frame_stack=4, frame_skip=4, screen_size=84, max_episode_steps=108_000):
        self.screen_size = screen_size
        self.frame_stack = frame_stack
        self._frames = deque(maxlen=frame_stack)
        super().__init__(env_id, frame_skip=frame_skip, max_episode_steps=max_episode_steps)
        self.obs_kind = "image"
        self.row_count = 6
        self.column_count = 6

    def _preprocess(self, frame):
        import cv2
        if frame.ndim == 3:
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        return cv2.resize(frame, (self.screen_size, self.screen_size), interpolation=cv2.INTER_AREA).astype(np.uint8)

    def get_initial_state(self):
        self._frames.clear()
        obs, _ = self.env.reset()
        frame = self._preprocess(np.asarray(obs))
        for _ in range(self.frame_stack):
            self._frames.append(frame)
        self._last_reward = 0.0
        self._last_done = False
        return np.stack(list(self._frames), axis=0)

    def get_next_state(self, state, action, player=1):
        obs = state
        total_reward = 0.0
        terminated = truncated = False
        for _ in range(self.frame_skip):
            raw, reward, terminated, truncated, _ = self.env.step(int(action))
            total_reward += float(reward)
            if terminated or truncated:
                break
        self._last_reward = total_reward
        self._last_done = terminated or truncated
        frame = self._preprocess(np.asarray(raw))
        self._frames.append(frame)
        return np.stack(list(self._frames), axis=0)

    def get_encoded_state(self, state):
        frames = np.asarray(state, dtype=np.float32)
        return frames / 255.0


In [ ]:
# Requires: pip install "gymnasium[atari,accept-rom-license]" opencv-python
# First run may prompt to accept the Atari ROM license.

try:
    atari = AtariGym("ALE/Pong-v5")
    state = atari.get_initial_state()
    print(atari)
    print("stacked frames shape:", state.shape)
    encoded = atari.get_encoded_state(state)
    print("encoded shape (C,H,W):", encoded.shape)

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, atari.frame_stack, figsize=(12, 3))
    for i, ax in enumerate(axes):
        ax.imshow(state[i], cmap="gray")
        ax.set_title(f"frame t-{atari.frame_stack - 1 - i}")
        ax.axis("off")
    plt.suptitle("MuZero Atari observation (stacked grayscale)")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Atari demo skipped:", exc)
    print("Install Atari dependencies to run this cell.")
